In [ ]:
# to fix imports 
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Plan

Column Cleaning
1. choose relevant columns first based on the goal
    - get guide columns, compare to actual column
2. inspect nutriments (contains cluster fields)
3. sanity checks for the nutriments
4. other

## Columns found

**Clustering inputs (per 100 g)**
- [x] nutriment.sugars
- [x] nutriment.fat
- [x] nutriment.energy_kcal
- [x] nutriment.sodium

rule: NaN if invalid values that dropping the whole row

## Unchecked

**Post-hoc check only: Nutri-Score**
- [ ] nutriscore_grade (str)
- [ ] nutriscore_score (float64)

**Association rule mining: ingredient tags**
- [ ] ingredients_tags (object)
- [ ] ingredients_original_tags
- [ ] ingredients_text
- [ ] ingredients_n

**Association rule mining: additive tags**
- [ ] additives_tags (object)
- [ ] additives_n
- [ ] new_additives_n

**categories**
- [ ] categories (str)
- [ ] categories_tags (object)
- [ ] food_groups_tags
- [ ] categories_properties

## 1. Columns named by the given guide

the following are from the guide

**Clustering inputs (per 100 g)**
- sugar
- sodium
- fat
- energy

**Post-hoc check only**
- Nutri-Score

**Association rule mining**
- ingredient tags
- additive tags

**Mentioned, no stated use**
- categories

**Row identity**
- product (one row per product)

### Inspect columns that matches

In [ ]:
# setup
import pandas as pd
from src.config import PH_RAW_PARQUET_PATH
pd.set_option("display.max_rows", None)
dataframe = pd.read_parquet(PH_RAW_PARQUET_PATH)

In [ ]:
print(dataframe.dtypes.to_string())

findings:

nutri-score
- nutriscore_grade (str), the categorical label
- nutriscore_score (float64), the numeric score the grade comes from

ingredient tags
- ingredients_tags (object)
- related: ingredients_original_tags, ingredients_text, ingredients_n

additive tags
- additives_tags (object)
- related: additives_n, new_additives_n

categories
- categories (str), categories_tags (object)
- related: food_groups_tags, categories_properties

row identity
- code (str), duplicate check belongs to row cleaning

missing
- sugar, sodium, fat, and energy have no flat columns. there is no *_100g columns

no nutrition values. lets try scanning for keywords

In [ ]:
keywords = [
    "sugar", "sodium", "fat", "energy", # matching keywords
    "nutriment", "kcal", "kj"
]
matches = [
    column
    for column in dataframe.columns
    if any(keyword in column.lower() for keyword in keywords)
]
print(matches)

nutriments shows up. from all rows, collect all values and count them

In [ ]:
from collections import Counter

name_counts = Counter()
for nutriments in dataframe["nutriments"].dropna():
    for entry in nutriments:
        name_counts[entry["name"]] += 1

for name, count in sorted(name_counts.most_common(40)):
    print(name, count)


recap: values to look for:
- sugar
- sodium
- fat
- energy


we found:
(format: <value>: <amount found in rows>)

sugar:
sugars 3859
added-sugars 943

sodium:
sodium 4020

fat:
fat 4565

energy:
energy 4579
energy-kcal 4572
energy-kj 2414

---

decision:
sugar = nutriment.sugars (added sugars is just 943 rows, reducing our dataset)
sodium = nutriment.sodium
fat = nutriment.fat
energy = undecided


In [ ]:
import numpy as np
import pandas as pd


def flatten_nutriments(nutriments):
    row = {}
    if not isinstance(nutriments, (list, np.ndarray)):
        return row
    for entry in nutriments:
        row[entry["name"]] = entry["100g"]
    return row


records = []
for nutriments in dataframe["nutriments"]:
    records.append(flatten_nutriments(nutriments))

nutrient_df = pd.DataFrame(records, index=dataframe.index)
print(nutrient_df.shape)

decide for energy, inspect their values

In [ ]:
check = nutrient_df[["energy", "energy-kcal", "energy-kj"]].dropna(subset=["energy", "energy-kcal"])
ratio = check["energy"] / check["energy-kcal"]
print(ratio.describe())

both = nutrient_df[["energy", "energy-kj"]].dropna()
print((both["energy"] == both["energy-kj"]).mean())

verdict:
all three say the same value but in different units:
energy and energy-kj are kJ (99.9% identical)
energy-kcal is kcal (energy / energy-kcal is about 4.184)

decision:
use energy-kcal only
energy and energy-kj are dropped, they are duplicates
keeping more than one would count energy twice in the clustering

now lets grab all the rows that has
sugars, sodium, fat, energy-kcal

In [ ]:
final_columns = ["sugars", "sodium", "fat", "energy-kcal"]
modeling_df = nutrient_df[final_columns]
print(modeling_df.notna().mean())
print(modeling_df.notna().all(axis=1).sum())

this tells us that around 45% (0.45) of the rows have these four keys

sanity check for invalid inputs

In [ ]:
final_columns = ["sugars", "sodium", "fat", "energy-kcal"]
modeling_df = nutrient_df[final_columns].copy()

print(modeling_df.describe())

rules = {
    "negative": (modeling_df < 0).any(axis=1),
    "sugars_over_100": modeling_df["sugars"] > 100,
    "fat_over_100": modeling_df["fat"] > 100,
    "sodium_over_40": modeling_df["sodium"] > 40,
    "kcal_over_950": modeling_df["energy-kcal"] > 950,
    "macros_over_105": (
        nutrient_df["fat"] + nutrient_df["sugars"] + nutrient_df["proteins"]
    ) > 105,
}

for name, mask in rules.items():
    print(name, mask.sum())

invalid values found, lets inspect them

In [ ]:
flagged = (
    nutrient_df.loc[any_flag, ["sugars", "sodium", "fat", "energy-kcal", "proteins"]]
    .join(dataframe.loc[any_flag, ["code", "product_name"]])
    .rename(columns={
        "energy-kcal": "kcal",
        "product_name": "product",
    })
    .sort_values("sodium", ascending=False)
)

print(f"Flagged products: {len(flagged)}")
print(flagged.round(1).to_string(index=False))


verdict:
the flagged values are impossible or inconsistent, so they are set to NaN
only the bad value is set to NaN, the row is kept
the missing-data step handles all gaps in one place later

rules applied:
sugars over 100: NaN
fat over 100: NaN
sodium over 40: NaN
energy-kcal over 950: NaN (cutoff raised from 900, pure oils are valid at about 930)
energy-kcal is 0 while fat is above 1: NaN (zero means not entered)

kept as valid:
canola oil and corn oil (fat 100, kcal about 930)

note:
sodium over 40 is the largest group (27 rows), most look like mg typed in a g field
these are set to NaN for now, dividing by 1000 is kept as an alternative for the stability test
sodium errors below 40 (for example 4.8 or 11.6 on a milk tea) are not caught by this rule
all zero rows (for example knorr sinigang) are not caught either, they look like missing data
recount complete records after applying

moving to other columns now:


check empty or NaN tags and junk grades
check language prefixes in ingredients_tags
check e-number overlap with additives_tags

In [ ]:
def tag_length(tags):
    if isinstance(tags, (list, np.ndarray)):
        return len(tags)
    return 0


print(dataframe["nutriscore_grade"].value_counts(dropna=False))

for column in ["ingredients_tags", "additives_tags", "categories_tags"]:
    lengths = dataframe[column].apply(tag_length)
    print(column, "empty or NaN:", (lengths == 0).sum())
    print(lengths.describe())

from collections import Counter

prefix_counts = Counter()
tag_counts = Counter()
for tags in dataframe["ingredients_tags"]:
    if not isinstance(tags, (list, np.ndarray)):
        continue
    for tag in tags:
        prefix_counts[tag.split(":")[0]] += 1
        tag_counts[tag] += 1

print(prefix_counts.most_common(10))
print(tag_counts.most_common(30))